# Spark performance

Uses synthetic data only; it does not read or modify the AdventureWorks Gold tables. Run after rebuilding a Databricks workspace. The default is intentionally small. Increase the row count only after confirming cluster cost and capacity. Timings on this dataset are educational observations, not production benchmarks.

In [ ]:
from pyspark.sql import functions as F
import time
import uuid

lab_rows = 100_000
lab_key_count = 1_000

facts = (
    spark.range(lab_rows)
    .withColumn(
        "join_key",
        F.when(F.col("id") < int(lab_rows * 0.8), F.lit(0))
        .otherwise((F.col("id") % lab_key_count).cast("long"))
    )
    .withColumn("amount", (F.col("id") % 1000).cast("double") / 100)
)

dimension = spark.range(lab_key_count).select(
    F.col("id").alias("join_key"),
    F.concat(F.lit("key-"), F.col("id").cast("string")).alias("label")
)

print("Synthetic fact rows:", lab_rows)
print("Initial fact partitions:", facts.rdd.getNumPartitions())
print("Configured shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

In [ ]:
skew_counts = facts.groupBy("join_key").count().orderBy(F.desc("count"))
display(skew_counts.limit(10))

wide_shuffle = (
    facts.repartition(200, "join_key")
    .groupBy("join_key")
    .agg(F.sum("amount").alias("total_amount"))
)

started = time.perf_counter()
wide_shuffle_rows = wide_shuffle.count()
wide_shuffle_seconds = time.perf_counter() - started
print("Wide-shuffle output rows:", wide_shuffle_rows)
print("Wide-shuffle elapsed seconds:", round(wide_shuffle_seconds, 3))
wide_shuffle.explain("formatted")

In [ ]:
automatic_join = facts.join(dimension, on="join_key", how="inner")
broadcast_join = facts.join(F.broadcast(dimension), on="join_key", how="inner")
merge_join = facts.hint("merge").join(
    dimension.hint("merge"), on="join_key", how="inner"
)

for label, joined in [
    ("automatic", automatic_join),
    ("broadcast hint", broadcast_join),
    ("merge hint", merge_join),
]:
    started = time.perf_counter()
    result_count = joined.count()
    elapsed = time.perf_counter() - started
    print(label, "rows:", result_count, "seconds:", round(elapsed, 3))
    joined.explain("formatted")

In [ ]:
repartitioned = facts.repartition(8, "join_key")
coalesced = facts.coalesce(8)
print("Repartitioned partition count:", repartitioned.rdd.getNumPartitions())
print("Coalesced partition count:", coalesced.rdd.getNumPartitions())

for label, candidate in [("repartition", repartitioned), ("coalesce", coalesced)]:
    started = time.perf_counter()
    candidate_rows = candidate.count()
    print(label, "rows:", candidate_rows, "seconds:", round(time.perf_counter() - started, 3))

cached = facts.cache()
try:
    started = time.perf_counter()
    print("Cached first count:", cached.count(), "seconds:", round(time.perf_counter() - started, 3))
    started = time.perf_counter()
    print("Cached repeat count:", cached.count(), "seconds:", round(time.perf_counter() - started, 3))
finally:
    cached.unpersist()

In [ ]:
lab_path = f"dbfs:/tmp/aw_spark_performance_lab_{uuid.uuid4().hex}"
many_files_path = lab_path + "/many_files"
one_file_path = lab_path + "/one_file"

try:
    facts.repartition(8).write.mode("overwrite").parquet(many_files_path)
    facts.coalesce(1).write.mode("overwrite").parquet(one_file_path)

    for label, path in [("repartition(8)", many_files_path), ("coalesce(1)", one_file_path)]:
        part_files = [item for item in dbutils.fs.ls(path) if item.name.startswith("part-")]
        print(label, "files:", len(part_files), "bytes:", sum(item.size for item in part_files))
finally:
    dbutils.fs.rm(lab_path, recurse=True)

In [ ]:
aqe_setting = "spark.sql.adaptive.enabled"
original_aqe_setting = spark.conf.get(aqe_setting)
try:
    spark.conf.set(aqe_setting, "true")
    aqe_join = facts.repartition(200, "join_key").join(
        dimension.repartition(200, "join_key"), on="join_key", how="inner"
    )
    print("AQE enabled:", spark.conf.get(aqe_setting))
    print("AQE join rows:", aqe_join.count())
    aqe_join.explain("formatted")
finally:
    spark.conf.set(aqe_setting, original_aqe_setting)

## Optional Delta layout experiment

If the rebuilt workspace permits temporary Delta paths and `OPTIMIZE`, write the synthetic data to a unique temporary Delta path, record file counts and query plans, run `OPTIMIZE` with `ZORDER BY (join_key)`, compare results, and delete only that unique temporary path. Do not run these commands against the AdventureWorks Gold tables. Record cluster settings and input size; skip the experiment if the target workspace does not permit the required temporary storage.